# Notebook 43 — Adaptive Resource Allocation v3

**Operating regimes specify controller policies. Controller policies specify resource allocation.**

Notebook 37 closed the first systems arc by turning confidence scheduling, throughput objectives, and hardware constraints into deployment-scale operating regimes. Notebook 43 starts the second systems arc: a controller that spends scarce serving resources according to the active regime.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
from matplotlib.colors import ListedColormap, BoundaryNorm
from zipfile import ZipFile

FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 160,
    "font.size": 12,
    "axes.titlesize": 24,
    "axes.labelsize": 16,
    "legend.fontsize": 11,
})

COLORS = {
    "balanced": "tab:blue",
    "compute-limited": "tab:orange",
    "memory-limited": "tab:green",
    "latency-protected": "tab:red",
}

def savefig(name):
    path = FIG_DIR / name
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    return path

def box(ax, xy, w, h, text, fontsize=13, lw=1.8, fc="white"):
    x, y = xy
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.035",
        linewidth=lw, edgecolor="black", facecolor=fc
    )
    ax.add_patch(patch)
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fontsize)
    return patch

def arrow(ax, start, end, lw=1.8, mutation_scale=14, connectionstyle="arc3"):
    arr = FancyArrowPatch(
        start, end,
        arrowstyle="->",
        linewidth=lw,
        mutation_scale=mutation_scale,
        color="black",
        connectionstyle=connectionstyle
    )
    ax.add_patch(arr)
    return arr

## 01 — Notebook transition

The first arc identified mechanisms and constraints. The second arc begins when those constraints become controller inputs.

\[
\rho_t=f(s_t)
\]

\[
b_t=\pi(\rho_t,s_t)
\]

\[
a_t=A(b_t)
\]

\[
y_t=g(a_t)
\]

where \(s_t\) is measured serving state, \(\rho_t\) is the active operating regime, \(b_t\) is the resource-budget vector, \(a_t\) is the allocation action, and \(y_t\) is the observed serving outcome.

In [ ]:
fig, ax = plt.subplots(figsize=(13.5, 6.5))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Resource allocation starts the second systems arc", pad=20)

first = [
    ("00\nContext", 0.04),
    ("07\nVerification\nResource", 0.20),
    ("13\nConfidence\nScheduling", 0.36),
    ("17\nSemi-AR\nDrafting", 0.52),
    ("23\nThroughput\nObjective", 0.68),
    ("29\nHardware\nConstraints", 0.84),
    ("37\nOperating\nRegimes", 0.96),
]
y1 = 0.68
for label, x in first:
    w = 0.11 if x < 0.9 else 0.105
    box(ax, (x - w/2, y1 - 0.055), w, 0.11, label, fontsize=10)
for (_, x0), (_, x1) in zip(first[:-1], first[1:]):
    arrow(ax, (x0 + 0.058, y1), (x1 - 0.058, y1), mutation_scale=11)

ax.plot([0.05, 0.95], [0.50, 0.50], color="black", lw=1.4)
ax.text(0.50, 0.53, "first arc: mechanism → operating regime", ha="center", va="bottom", fontsize=13)

second = [
    ("43\nResource\nAllocation", 0.42),
    ("49\nAdaptive\nInfrastructure", 0.57),
    ("53\nDistributed\nScheduling", 0.72),
]
y2 = 0.32
for label, x in second:
    box(ax, (x - 0.065, y2 - 0.055), 0.13, 0.11, label, fontsize=10)
for (_, x0), (_, x1) in zip(second[:-1], second[1:]):
    arrow(ax, (x0 + 0.068, y2), (x1 - 0.068, y2), mutation_scale=11)

ax.text(0.57, 0.13, "second arc: operating regime → resource allocation → infrastructure",
        ha="center", va="center", fontsize=14)
savefig("43_repository_second_arc.png")

## 02 — Hero controller

The controller separates four operations:

1. estimate the current serving state;
2. classify the operating regime;
3. choose a budget policy;
4. allocate scarce resources to verification, compute, memory, and reserve capacity.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Adaptive resource-allocation controller", pad=20)

box(ax, (0.07, 0.74), 0.17, 0.08, "Incoming\nrequests", fontsize=12)
box(ax, (0.35, 0.74), 0.17, 0.08, "State\nestimator", fontsize=12)
box(ax, (0.35, 0.58), 0.22, 0.09, "Operating-regime\nclassifier", fontsize=12)
box(ax, (0.38, 0.42), 0.16, 0.08, "Budget\npolicy", fontsize=12)
box(ax, (0.38, 0.27), 0.16, 0.08, "Adaptive\nallocator", fontsize=12)

resources = [
    ("compute", 0.16),
    ("memory", 0.34),
    ("verify", 0.52),
    ("reserve", 0.70),
    ("latency", 0.86),
]
for name, x in resources:
    box(ax, (x - 0.06, 0.10), 0.12, 0.07, name, fontsize=10)

box(ax, (0.38, -0.02), 0.20, 0.08, "Serving\nsystem", fontsize=12)
box(ax, (0.73, 0.08), 0.18, 0.08, "Observed\nsystem state", fontsize=12)
box(ax, (0.07, 0.38), 0.17, 0.08, "State\nupdate", fontsize=12)

arrow(ax, (0.24, 0.78), (0.35, 0.78))
arrow(ax, (0.435, 0.74), (0.46, 0.67))
arrow(ax, (0.46, 0.58), (0.46, 0.50))
arrow(ax, (0.46, 0.42), (0.46, 0.35))
for _, x in resources:
    arrow(ax, (0.46, 0.27), (x, 0.17), lw=1.2, mutation_scale=10)
    arrow(ax, (x, 0.10), (0.48, 0.06), lw=1.1, mutation_scale=9)

arrow(ax, (0.58, 0.02), (0.73, 0.10))
arrow(ax, (0.82, 0.16), (0.48, 0.74), connectionstyle="arc3,rad=0.15")
arrow(ax, (0.16, 0.46), (0.16, 0.74))
arrow(ax, (0.16, 0.38), (0.16, 0.06), connectionstyle="arc3,rad=-0.08")
arrow(ax, (0.24, 0.40), (0.39, 0.29), connectionstyle="arc3,rad=-0.05")

ax.text(0.50, 0.93, r"$\rho_t=f(s_t),\quad b_t=\pi(\rho_t,s_t),\quad a_t=A(b_t)$",
        ha="center", va="center", fontsize=17)
ax.text(0.50, 0.02, "Operating regimes specify adaptive resource allocation.",
        ha="center", va="bottom", fontsize=14)
savefig("43_controller_architecture.png")

## 03 — Batch admission under resource constraints

A serving controller does not only choose a verification length. It also decides how much work enters the active batch.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6.4))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Batch admission control under resource constraints", pad=18)

box(ax, (0.31, 0.76), 0.42, 0.09, "Admission depends on the current resource budget.", fontsize=13)

main = [
    ("Incoming\nrequests", 0.08),
    ("Queue", 0.28),
    ("Admission /\nallocation\ncontroller", 0.50),
    ("Admitted\nbatch", 0.72),
    ("Target\nverify", 0.90),
]
for label, x in main:
    w = 0.14 if x != 0.50 else 0.18
    h = 0.10 if x != 0.50 else 0.15
    box(ax, (x - w/2, 0.55 - h/2), w, h, label, fontsize=12)
for (_, x0), (_, x1) in zip(main[:-1], main[1:]):
    arrow(ax, (x0 + 0.07, 0.55), (x1 - 0.07, 0.55))

inputs = [
    ("compute\nbudget", 0.34),
    ("memory\nbudget", 0.45),
    ("latency\ntarget", 0.56),
    ("reserve\ncapacity", 0.67),
]
for label, x in inputs:
    box(ax, (x - 0.055, 0.19), 0.11, 0.08, label, fontsize=10)
    arrow(ax, (x, 0.27), (0.50, 0.475), lw=1.4, mutation_scale=11)

savefig("43_batch_admission.png")

## 04 — Regime-conditioned budget vectors

Each operating regime maps to a different budget partition. Balanced serving spends relatively evenly; latency-protected serving preserves reserve; memory-limited serving shifts budget into verification and compute that reduce memory load.

In [ ]:
regimes = ["balanced", "compute-limited", "memory-limited", "latency-protected"]
components = ["compute", "memory", "verify", "reserve"]
budgets = np.array([
    [0.30, 0.25, 0.30, 0.15],
    [0.22, 0.30, 0.20, 0.28],
    [0.32, 0.18, 0.25, 0.25],
    [0.25, 0.22, 0.18, 0.35],
])
component_colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_title("Adaptive budget partitions by operating regime", pad=18)

y = np.arange(len(regimes))
left = np.zeros(len(regimes))
for k, comp in enumerate(components):
    ax.barh(y, budgets[:, k], left=left, height=0.65, label=comp, color=component_colors[k])
    for i, v in enumerate(budgets[:, k]):
        ax.text(left[i] + v/2, y[i], f"{comp}\n{v:.2f}", ha="center", va="center", fontsize=10)
    left += budgets[:, k]

ax.set_yticks(y)
ax.set_yticklabels(regimes)
ax.set_xlim(0, 1)
ax.set_xlabel("normalized resource budget")
ax.grid(axis="x", alpha=0.3)
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.20), ncol=4)
savefig("43_budget_partitions.png")

## 05 — Policy phase diagram

The controller selects a policy from measurable resource state. This is the v3 consolidation: one phase diagram replaces multiple overlapping controller maps.

In [ ]:
x = np.linspace(0.1, 1.0, 160)  # compute availability
y = np.linspace(0.1, 1.0, 160)  # memory availability
X, Y = np.meshgrid(x, y)

# Four policy labels:
# 0 balanced, 1 compute-limited, 2 memory-limited, 3 latency-protected
balanced_score = np.exp(-((X-0.55)**2/0.055 + (Y-0.52)**2/0.070))
compute_pressure = 1 - X
memory_pressure = 1 - Y
latency_pressure = 0.55*X + 0.75*Y

Z = np.zeros_like(X, dtype=int)
Z[compute_pressure > 0.55] = 1
Z[memory_pressure > 0.58] = 2
Z[latency_pressure > 1.05] = 3
Z[balanced_score > 0.42] = 0

cmap = ListedColormap([COLORS["balanced"], COLORS["compute-limited"], COLORS["memory-limited"], COLORS["latency-protected"]])
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], cmap.N)

fig, ax = plt.subplots(figsize=(10.5, 8))
im = ax.imshow(Z, origin="lower", extent=[0.1, 1.0, 0.1, 1.0], cmap=cmap, norm=norm, aspect="auto")
ax.set_title("Allocation-policy phase diagram", pad=18)
ax.set_xlabel("compute availability")
ax.set_ylabel("memory availability")

ax.text(0.45, 0.35, "balanced", fontsize=14)
ax.text(0.22, 0.75, "compute-\nlimited", fontsize=13)
ax.text(0.73, 0.28, "memory-\nlimited", fontsize=13)
ax.text(0.28, 0.24, "latency-\nprotected", fontsize=13)
ax.text(0.58, 0.88, "verification-\nheavy", fontsize=13)

cbar = fig.colorbar(im, ax=ax, ticks=[0, 1, 2, 3])
cbar.ax.set_yticklabels(["balanced", "compute-limited", "memory-limited", "latency-protected"])
cbar.set_label("selected controller policy")
savefig("43_policy_phase_diagram.png")

## 06 — Allocation surface

The resource surface shows why the controller should not spend resources uniformly: the marginal throughput gain depends jointly on compute and memory budgets.

In [ ]:
x = np.linspace(0.1, 1.0, 120)
y = np.linspace(0.1, 1.0, 120)
X, Y = np.meshgrid(x, y)

surface = (1 - np.exp(-2.4*X)) * (1 - np.exp(-2.2*Y))
surface *= (1 - 0.10*np.exp(-((X-0.55)**2 + (Y-0.50)**2)/0.08))
surface += 0.08*np.exp(-((X-0.86)**2/0.055 + (Y-0.88)**2/0.070))

points = {
    "balanced": (0.62, 0.62),
    "compute-limited": (0.42, 0.70),
    "memory-limited": (0.72, 0.42),
    "latency-protected": (0.55, 0.50),
}

fig, ax = plt.subplots(figsize=(10.5, 8))
cs = ax.contourf(X, Y, surface, levels=40)
ax.contour(X, Y, surface, levels=8, alpha=0.25)
for name, (px, py) in points.items():
    ax.scatter(px, py, s=95, color=COLORS[name], zorder=5)
    ax.text(px+0.02, py+0.02, name, fontsize=12)

ax.set_title("Resource allocation surface for adaptive serving", pad=18)
ax.set_xlabel("compute budget")
ax.set_ylabel("memory budget")
cbar = fig.colorbar(cs, ax=ax)
cbar.set_label("throughput proxy")
savefig("43_allocation_surface.png")

## 07 — Adaptive policy curves

Policy fractions vary continuously with available budget. In v3, these curves are the lightweight bridge from regime labels to actionable allocations.

In [ ]:
b = np.linspace(0.05, 1.0, 10)
compute_fraction = 0.18 + 0.55*(1-np.exp(-2.2*b))
verify_fraction = 0.16 + 0.45*(1-np.exp(-2.8*b)) - 0.03*np.maximum(b-0.75, 0)
memory_fraction = 0.36 - 0.12*b + 0.10*np.exp(-((b-0.32)**2)/0.035)
reserve_fraction = 1 - (compute_fraction + verify_fraction + memory_fraction)
reserve_fraction = np.clip(reserve_fraction, 0.08, 0.40)

fig, ax = plt.subplots(figsize=(11.5, 7))
ax.plot(b, compute_fraction, marker="o", label="compute fraction")
ax.plot(b, verify_fraction, marker="o", label="verification fraction")
ax.plot(b, memory_fraction, marker="o", label="memory fraction")
ax.plot(b, reserve_fraction, marker="o", label="reserve fraction")
ax.set_title("Adaptive policy curves allocate budget by regime", pad=18)
ax.set_xlabel("available resource budget")
ax.set_ylabel("allocation fraction")
ax.set_ylim(0, 0.85)
ax.grid(alpha=0.3)
ax.legend()
savefig("43_adaptive_policy_curves.png")

## 08 — Adaptive gains

The controller matters when fixed allocations leave performance on the table. Different regimes have different gain curves, so one global resource rule is too blunt.

In [ ]:
r = np.linspace(0.05, 1.0, 10)
gains = {
    "balanced": 1.18*(1-np.exp(-4.2*r)),
    "compute-limited": 0.98*(1-np.exp(-2.1*r)),
    "memory-limited": 0.96*(1-np.exp(-3.2*r)),
    "latency-protected": 0.86*(1-np.exp(-5.0*r)) - 0.20*np.maximum(r-0.62, 0),
}

fig, ax = plt.subplots(figsize=(11.5, 7))
for name, vals in gains.items():
    ax.plot(r, vals, marker="o", label=name, color=COLORS[name])
ax.set_title("Adaptive throughput gains depend on regime", pad=18)
ax.set_xlabel("allocated resource budget")
ax.set_ylabel("throughput gain proxy")
ax.grid(alpha=0.3)
ax.legend()
savefig("43_adaptive_gains.png")

## 09 — Latency-memory tradeoff

The same allocation can have different latency cost depending on the regime. This plot preserves the most useful diagnostic from v2 while using the v3 vocabulary.

In [ ]:
m = {
    "balanced": np.linspace(0.38, 2.10, 9),
    "compute-limited": np.linspace(0.36, 2.05, 9),
    "memory-limited": np.linspace(0.44, 2.42, 9),
    "latency-protected": np.linspace(0.28, 1.52, 9),
}
lat = {
    "balanced": 0.22 + 0.85*m["balanced"],
    "compute-limited": 0.18 + 0.90*m["compute-limited"],
    "memory-limited": 0.12 + 0.50*m["memory-limited"],
    "latency-protected": 0.16 + 1.58*m["latency-protected"],
}
stars = {
    "balanced": 3,
    "compute-limited": 3,
    "memory-limited": 4,
    "latency-protected": 3,
}

fig, ax = plt.subplots(figsize=(11.5, 7))
for name in regimes:
    ax.plot(m[name], lat[name], marker="o", label=name, color=COLORS[name])
    k = stars[name]
    ax.scatter(m[name][k], lat[name][k], s=95, color=COLORS[name], zorder=5)
    ax.text(m[name][k]+0.04, lat[name][k]+0.04, rf"$\ell^*={k}$", fontsize=12)

ax.set_title("Latency-memory tradeoff depends on operating regime", pad=18)
ax.set_xlabel("memory pressure proxy")
ax.set_ylabel("latency proxy")
ax.grid(alpha=0.3)
ax.legend()
savefig("43_latency_memory_tradeoff.png")

## 10 — Controller flow

Notebook 43 ends with the minimal deployable chain: regime, controller, budget partition, verification scheduler, serving system.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Resource allocation flow for adaptive serving", pad=18)

items = [
    ("Operating\nregime", 0.12),
    ("Adaptive\ncontroller", 0.32),
    ("Budget\npartition", 0.52),
    ("Verification\nscheduler", 0.72),
    ("Serving\nsystem", 0.90),
]
for label, x in items:
    box(ax, (x-0.07, 0.55), 0.14, 0.12, label, fontsize=13)
for (_, x0), (_, x1) in zip(items[:-1], items[1:]):
    arrow(ax, (x0+0.07, 0.61), (x1-0.07, 0.61))

box(ax, (0.28, 0.18), 0.48, 0.11, "The controller spends scarce resources according to the active regime.", fontsize=13)
savefig("43_resource_allocation_flow.png")

## 11 — Summary

Notebook 43 v3 reduces the figure set and strengthens the abstraction:

\[
\text{system state}
\rightarrow
\text{operating regime}
\rightarrow
\text{controller policy}
\rightarrow
\text{resource budget}
\rightarrow
\text{resource allocation}
\rightarrow
\text{serving throughput}.
\]

This prepares Notebook 49: adaptive infrastructure that implements the controller.

In [ ]:
# Package generated figures for download.
zip_path = Path("43_adaptive_resource_allocation_v3_figures.zip")
with ZipFile(zip_path, "w") as zf:
    for p in sorted(FIG_DIR.glob("43_*.png")):
        zf.write(p, arcname=f"figures/{p.name}")
print(f"Wrote {zip_path}")